# 🚀 Video2Video – Lanzador Optimizado para Kaggle (GPU Dual T4 ×2)

### ⚙️ Configuración previa en la barra lateral derecha de Kaggle:
1. **Accelerator**: Selecciona **GPU T4 x2** (te otorga 32 GB VRAM combinada).
2. **Internet**: Asegúrate de que esté en **Internet on** (requiere verificación telefónica gratuita de Kaggle).
3. **Secrets**: En `Add-ons` -> `Secrets`, agrega tus claves:
   - `GHP_GITHUB_TOKEN` *(opcional si el repo es público, obligatorio si es privado)*
   - `GOOGLE_API_KEY` *(para Gemini Flash / traducción y títulos virales)*
   - `FISH_API_KEY` *(para síntesis de voz Fish Audio TTS)*

   - `HF_TOKEN` *(obligatorio para Pyannote Diarization 3.1)*

In [1]:
# @title ⚡ Arrancar V2V + OpenReel (Kaggle & Google Colab Híbrido)
# @markdown **Elige el entorno de trabajo:**
# @markdown - `Web (Navegador)`: Abre OpenReel completo en tu navegador web.
# @markdown - `Desktop (App Local PC)`: Ejecuta solo el Backend GPU en Kaggle y conéctalo a OpenReel Desktop en tu PC.
MODO = "Desktop (App Local PC)"  # @param ["Web (Navegador)", "Desktop (App Local PC)"]
IS_DESKTOP_MODE = "Desktop" in MODO
import os, time, threading, sys, warnings, subprocess, re, urllib.request
from pathlib import Path
warnings.filterwarnings("ignore")

# Detección automática del entorno (Google Colab vs Kaggle)
IS_COLAB = ("google.colab" in sys.modules) or (Path("/content").exists() and not Path("/kaggle").exists())
IS_KAGGLE = Path("/kaggle").exists()
PLATFORM_NAME = "GOOGLE COLAB" if IS_COLAB else "KAGGLE"

print("=" * 65)
print(f"🎬 INICIANDO VIDEO2VIDEO EN {PLATFORM_NAME}")
print("=" * 65)

# --- 1. HARDWARE & DUAL-GPU ---
try:
    import torch
    gpu_count = torch.cuda.device_count()
    print(f"🔥 GPUs NVIDIA detectadas: {gpu_count}")
    for i in range(gpu_count):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")
    os.environ["V2V_DEVICE"] = "cuda:0"
    os.environ["V2V_DEVICE_SECONDARY"] = "cuda:1" if gpu_count >= 2 else "cuda:0"
    print("⚡ Modo Dual-GPU activado" if gpu_count >= 2 else "⚡ Modo Single-GPU activado")
except Exception as e:
    print("⚠️ Error comprobando GPUs:", e)

# --- 2. SECRETOS (SOPORTE COLAB & KAGGLE) ---
SECRET_KEYS = ["GHP_GITHUB_TOKEN", "FISH_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"]

# 2.1 Intentar Google Colab userdata
try:
    from google.colab import userdata
    for name in SECRET_KEYS:
        try:
            val = userdata.get(name)
            if val:
                val_str = str(val).strip()
                os.environ[name] = val_str
                masked = f"...{val_str[-4:]}" if len(val_str) >= 4 else "***"
                print(f"🔑 Secreto cargado (Colab): {name} ({masked})")
        except Exception:
            pass
except ImportError:
    pass

# 2.2 Intentar Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for name in SECRET_KEYS:
        try:
            val = user_secrets.get_secret(name)
            if val and not os.getenv(name):
                val_str = str(val).strip()
                os.environ[name] = val_str
                masked = f"...{val_str[-4:]}" if len(val_str) >= 4 else "***"
                print(f"🔑 Secreto cargado (Kaggle): {name} ({masked})")
        except Exception:
            pass
except Exception:
    pass

if os.getenv("HF_TOKEN"):
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"], write_permission=False)
        print("🤗 Hugging Face autenticado con éxito.")
    except Exception:
        pass

# --- 3. REPOSITORIO & VARIABLES ---
REPO_PATH = Path("/content/v2v") if IS_COLAB else Path("/kaggle/working/v2v")
REPO_PATH.parent.mkdir(parents=True, exist_ok=True)

GHP_TOKEN = os.getenv("GHP_GITHUB_TOKEN", "").strip()
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git" if GHP_TOKEN else "https://github.com/almagomen/v2v.git"
BRANCH_NAME = "monorepo-subtree"
BACKEND_PORT, FRONTEND_PORT = 8000, 5173

if not REPO_PATH.exists():
    print(f"📥 Clonando repositorio en {REPO_PATH}...")
    res_clone = subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL, str(REPO_PATH)], capture_output=True, text=True)
    if res_clone.returncode != 0:
        err_out = (res_clone.stderr or res_clone.stdout or '').strip()
        print(f"\n❌ Error al clonar repositorio:\n{err_out}")
        if not GHP_TOKEN:
            print("\n⚠️ IMPORTANTE: Si el repositorio es privado, debes agregar tu token de GitHub 'GHP_GITHUB_TOKEN'.")
            if IS_COLAB:
                print("👉 En Google Colab: Haz clic en el ícono de la llave 🔑 (Secretos) a la izquierda, agrega 'GHP_GITHUB_TOKEN' y activa el interruptor 'Acceso del notebook'.")
            else:
                print("👉 En Kaggle: Ve a 'Add-ons' -> 'Secrets' y agrega 'GHP_GITHUB_TOKEN'.")
        raise RuntimeError(f"git clone fallo: {err_out}")
    print("✔️ Repositorio clonado con éxito.")
else:
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "origin", BRANCH_NAME, "-q"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{BRANCH_NAME}", "-q"], check=True)
    subprocess.run(f"find {REPO_PATH} -name '__pycache__' -exec rm -rf {{}} + 2>/dev/null || true", shell=True)
    print("✔️ Repositorio actualizado al último commit de GitHub.")
    # Invalidar caché de módulos en memoria para reflejar cambios sin reiniciar kernel
    for _m in list(sys.modules.keys()):
        if _m.startswith('src.') or _m == 'src' or _m.startswith('api.') or _m == 'api':
            del sys.modules[_m]

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Sincronizar secretos en .env para Vite (en openreel/apps/web/ para evitar commits accidentales)
_vite_env_path = REPO_PATH / "openreel" / "apps" / "web" / ".env"
env_lines = [f"{k}={os.getenv(k)}\n" for k in ["GOOGLE_API_KEY", "FISH_API_KEY", "HF_TOKEN"] if os.getenv(k)]
if env_lines:
    try:
        _vite_env_path.parent.mkdir(parents=True, exist_ok=True)
        _vite_env_path.write_text("".join(env_lines), encoding="utf-8")
    except Exception:
        pass
# --- 4. DEPENDENCIAS Y MODELOS EN GPU ---
if 'models_loaded' not in globals():
    print("📦 Instalando dependencias con uv...")
    _deno_cmd = "(which deno >/dev/null || (curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1 && cp -f /root/.deno/bin/deno /usr/local/bin/))"
    _pnpm_setup = f"npm install -g pnpm --silent >/dev/null 2>&1 && {_deno_cmd}" if not IS_DESKTOP_MODE else _deno_cmd
    subprocess.run(f"pip install -q uv && {_pnpm_setup}", shell=True)

    # 1. Paquetes con dependencias
    subprocess.run(
        "uv pip install 'yt-dlp[default]' bs-roformer-infer tinydb sentencex soundfile "
        "'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' "
        "'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' "
        "'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' 'google-genai>=2.9.0' -q ",
        shell=True,
    )

    # 2. Paquetes con --no-deps (evitan conflictos con PyTorch preinstalado en Kaggle)
    subprocess.run(
        "uv pip install --no-deps git+https://github.com/jianfch/stable-ts.git "
        "git+https://github.com/pyannote/pyannote-audio.git openai-whisper "
        "'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' "
        "'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' -q",
        shell=True,
    )

    # Binario Cloudflare Tunnel
    cf_bin = Path("/tmp/cloudflared")
    if not cf_bin.exists():
        print("🌐 Descargando Cloudflare Tunnel...")
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(cf_bin)], check=True)
        subprocess.run(["chmod", "+x", str(cf_bin)], check=True)

    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import preload_roformer_models

    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': preload_roformer_models(),
        'models_loaded': True
    })
    print("✅ Modelos inicializados en memoria.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 5. SERVIDOR BACKEND (Uvicorn en Background) ---
import uvicorn
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    global server
    for m in [m for m in list(sys.modules.keys()) if m.startswith(('api', 'src')) and 'models' not in m]:
        del sys.modules[m]

    from api.main import app
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_level="warning", loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config)
    server.run()

if 'server' in globals() and server.started:
    print("🛑 Reiniciando servidor backend...")
    server.should_exit = True
    time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# Healthcheck: esperar que el backend responda antes de lanzar el frontend
print("⏳ Esperando que el backend responda en el puerto 8000...")
for _hc in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{BACKEND_PORT}/health", timeout=1) as _resp:
            if _resp.status == 200:
                print("✅ Backend activo.")
                break
    except Exception:
        pass
    time.sleep(1)
else:
    print("⚠️ Backend no respondió después de 60s. Continuando igualmente...")

# --- 6. FRONTEND (Vite / OpenReel) ---
if IS_DESKTOP_MODE:
    print("⚡ Modo Desktop: Kaggle operará exclusivamente como backend de GPU.")
    print("   (Saltando inicialización de Vite en Kaggle para ahorrar RAM y recursos)")
else:
    OPENREEL_PATH = REPO_PATH / "openreel"
    WEB_PATH = OPENREEL_PATH / "apps" / "web"

    vite_already_running = False
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=1) as resp:
            if resp.status in (200, 304):
                vite_already_running = True
    except Exception:
        pass

    if vite_already_running and globals().get('front_proc') and globals()['front_proc'].poll() is None:
        print("⚡ OpenReel (Vite) ya está activo. Reutilizando proceso.")
    else:
        print("🏗️ Preparando OpenReel...")
        # Smart pnpm install: skip si node_modules está actualizado
        _lock_file = OPENREEL_PATH / "pnpm-lock.yaml"
        _modules_mark = OPENREEL_PATH / "node_modules" / ".package-lock.json"
        if not _modules_mark.exists() or (_lock_file.exists() and _lock_file.stat().st_mtime > _modules_mark.stat().st_mtime):
            subprocess.run(["pnpm", "install", "--silent"], cwd=str(OPENREEL_PATH), check=True)
        else:
            print("⚡ node_modules al día. Saltando pnpm install.")
        subprocess.run(["fuser", "-k", f"{FRONTEND_PORT}/tcp"], stderr=subprocess.DEVNULL)
        
        globals()['front_proc'] = subprocess.Popen(
            ["pnpm", "exec", "vite", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
            cwd=str(WEB_PATH), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )

        # Healthcheck Vite
        print("⏳ Esperando que OpenReel responda en el puerto 5173...")
        for _ in range(45):
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{FRONTEND_PORT}/", timeout=1) as resp:
                    if resp.status in (200, 304):
                        print("✅ OpenReel Vite activo.")
                        break
            except Exception:
                pass
            time.sleep(1)

# --- 7. CLOUDFLARE TUNNEL (PERSISTENCIA TOTAL A NIVEL DE SO) ---
TARGET_PORT = BACKEND_PORT if IS_DESKTOP_MODE else FRONTEND_PORT
CF_URL_FILE = Path("/tmp/v2v_tunnel_url.txt")
CF_MODE_FILE = Path("/tmp/v2v_tunnel_mode.txt")

def check_cf_os_running():
    try:
        r = subprocess.run(["pgrep", "-f", "cloudflared tunnel"], capture_output=True, text=True)
        return r.returncode == 0
    except Exception:
        return False

# Preservar el túnel si cloudflared sigue corriendo en Linux para el mismo modo y puerto
is_running = check_cf_os_running()
has_saved_url = CF_URL_FILE.exists() and bool(CF_URL_FILE.read_text(encoding="utf-8").strip())
same_mode = CF_MODE_FILE.exists() and CF_MODE_FILE.read_text(encoding="utf-8").strip() == MODO

if is_running and has_saved_url and same_mode:
    tunnel_url = CF_URL_FILE.read_text(encoding="utf-8").strip()
    globals()['tunnel_url'] = tunnel_url
    print(f"♻️ Preservando túnel Cloudflare activo en el sistema: {tunnel_url}")
else:
    _target_desc = f"Backend FastAPI ({TARGET_PORT})" if IS_DESKTOP_MODE else f"OpenReel Web ({TARGET_PORT})"
    print(f"🌐 Levantando túnel HTTPS público para {_target_desc}...")
    subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
    time.sleep(0.5)
    cf_proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{TARGET_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    tunnel_url = None
    for _ in range(40):
        line = cf_proc.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(0.5)

    # Drenar Cloudflare stdout en background sin bloquear
    threading.Thread(target=lambda: [None for _ in iter(cf_proc.stdout.readline, '')], daemon=True).start()
    globals()['cf_proc'] = cf_proc
    globals()['tunnel_url'] = tunnel_url
    if tunnel_url:
        CF_URL_FILE.write_text(tunnel_url, encoding="utf-8")
        CF_MODE_FILE.write_text(MODO, encoding="utf-8")

print("\n" + "=" * 65)
if tunnel_url:
    if IS_DESKTOP_MODE:
        print("🎉 ¡BACKEND VIDEO2VIDEO LISTO PARA OPENREEL DESKTOP!")
        print("🔗 URL del Backend Remoto (Kaggle GPU Dual-T4):")
        print(f"👉 {tunnel_url}")
        print("\n📋 Instrucciones para OpenReel Desktop en tu PC:")
        print("1. Abre OpenReel Desktop en tu computadora.")
        print("2. Abre el panel Video2Video (botón en la barra superior o atajo Alt+V).")
        print("3. Haz clic en 'Tokens' -> 'URL del Backend Remoto'.")
        print(f"4. Pega la URL: {tunnel_url}")
        print("5. Haz clic en 'Probar conexión' y luego en 'Guardar y aplicar'.")
        print("6. ¡Listo! Todo el procesamiento se ejecutará en las GPUs Dual-T4 de Kaggle.")
    else:
        print("🎉 ¡ENTORNO LISTO Y FUNCIONANDO EN SEGUNDO PLANO!")
        print("🔗 Abre OpenReel en tu navegador:")
        print(f"👉 {tunnel_url}")
else:
    print("⚠️ No se pudo obtener la URL de Cloudflare automáticamente.")
print("=" * 65)


SyntaxError: invalid syntax (3496824902.py, line 162)

---
## 🔬 Prueba Paso a Paso del Pipeline (Diagnóstico Función por Función)
Ejecuta las siguientes celdas de forma individual para ver los logs que genera cada función exactamente igual que en el panel de V2V, midiendo los tiempos de cada etapa.

In [ ]:
# @title 🔗 0. Configurar URL a Probar
url = "https://www.youtube.com/watch?v=C0XGm2m5Yf4"  # @param {type:"string"}
print(f"🎯 URL seleccionada: {url}")


In [ ]:
# @title 📥 1. Descarga y Separación de Stems (Control Interactivo & Benchmark)
# @markdown El solapamiento está calibrado internamente en 2.0s con crossfade para garantizar una unión perfecta sin 'tics'.
modo_gpu = "Dual-GPU (GPU 0 + GPU 1)" # @param ["Dual-GPU (GPU 0 + GPU 1)", "Single-GPU (1 sola GPU)"]
forzar_reproceso = True # @param {type:"boolean"}

import time, os
from pathlib import Path
import soundfile as sf
import IPython.display as ipd
from src.download import download_video, separate_vocals

force_dual = (modo_gpu == "Dual-GPU (GPU 0 + GPU 1)")

print(f"⚙️ Analizando video: {url}")
video_path, info = download_video(url, force_new=False)

vocals_path = video_path.with_name("vocals.wav")
no_vocals_path = video_path.with_name("no_vocals.wav")

if forzar_reproceso:
    if vocals_path.exists(): vocals_path.unlink()
    if no_vocals_path.exists(): no_vocals_path.unlink()

print("=" * 60)
print("🎬 Iniciando separación con:")
print(f"   • Modo GPU: {'⚡ Dual-GPU (GPU 0 + GPU 1)' if force_dual else '🛡️ Single-GPU'}")
print("   • Solapamiento: 2.0s (fijo con crossfade anti-tic)")
print("=" * 60)

t0 = time.perf_counter()
voc_path, inst_path = separate_vocals(
    video_path,
    force_dual_gpu=force_dual,
)
t_elapsed = time.perf_counter() - t0

# Métricas usando soundfile sobre el archivo WAV extraído
audio_info = sf.info(str(voc_path))
dur_sec = audio_info.duration
sr = audio_info.samplerate
total_frames = audio_info.frames
speedup = dur_sec / t_elapsed if t_elapsed > 0 else 0

print("\n" + "=" * 60)
print("🏁 RESULTADOS DEL BENCHMARK:")
print(f"• Duración del audio: {dur_sec:.1f}s ({dur_sec/60:.2f} min)")
print(f"• Tiempo de separación: {t_elapsed:.1f} segundos")
print(f"• Velocidad relativa: {speedup:.1f}x tiempo real")
print("=" * 60)

# Reproductores interactivos de audio directamente desde disco
print("\n🎵 1. Pista de Voz Extraída (vocals.wav):")
ipd.display(ipd.Audio(str(voc_path)))

# Zoom de 10s en el punto de unión (midpoint) para verificar la transición limpia
if force_dual and dur_sec > 20:
    mid_sample = total_frames // 2
    win_samples = int(5.0 * sr)
    start_s = max(0, mid_sample - win_samples)
    end_s = min(total_frames, mid_sample + win_samples)
    zoom_data, _ = sf.read(str(voc_path), start=start_s, stop=end_s)
    if zoom_data.ndim == 2:
        zoom_data = zoom_data.T
    mid_sec = dur_sec / 2.0
    print(f"\n🔍 2. Zoom en el punto de unión (segundo {mid_sec-5:.1f} a {mid_sec+5:.1f}):")
    print("   (Verificación auditiva del empalme suave sin cortes ni tics):")
    ipd.display(ipd.Audio(zoom_data, rate=sr))

print("\n🎸 3. Pista Instrumental sin Voz (no_vocals.wav):")
ipd.display(ipd.Audio(str(inst_path)))


In [ ]:
# @title 📝 2. Transcripción (Whisper) y Diarización (Pyannote)
import time
from src.stt import transcribe_and_process

vocals_path = video_path.with_name("vocals.wav")
print(f"📝 Procesando STT + Diarización sobre: {vocals_path.name}\n")
t0 = time.perf_counter()
result, detected_lang, timings = transcribe_and_process(vocals_path)
t_step2 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 2 completado en {t_step2:.1f}s")
print(f"📊 Total segmentos detectados: {len(result)}")
print(f"🌐 Idioma original: {detected_lang}")
if result:
    print(f"🔹 Primer segmento: [{result[0]['start']:.2f}s - {result[0]['end']:.2f}s] {result[0]['speaker']}: {result[0]['text']}")
print("=" * 60)


In [ ]:
# @title 🌐 3. Traducción y Adaptación de Diálogos (Gemini)
import time
from src.translation import translate_text_list

texts_to_translate = [r["text"] for r in result]
src_lang = globals().get("detected_lang", "en")
print(f"🌐 Enviando {len(texts_to_translate)} frases a traducir ({src_lang} -> es)...\n")

t0 = time.perf_counter()
translated_texts = translate_text_list(
    texts_to_translate,
    target_lang="es",
    source_lang=src_lang,
    print_full_prompt=True
)
t_step3 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 3 completado en {t_step3:.1f}s")
if translated_texts:
    print(f"✨ Frases adaptadas con éxito: {len(translated_texts)}")
    print(f"   [Original 1]: {texts_to_translate[0]}")
    print(f"   [Adaptada 1]: {translated_texts[0]}")
else:
    print("❌ Falló la traducción.")
print("=" * 60)


In [ ]:
# @title ✨ 4. Generación de Título Viral (Gemini)
import time
from src.translation import generate_tiktok_title

# @markdown **Opciones de entrada:**
# @markdown Elige si generar el título basándote en los **diálogos** (transcripción completa) o en la **URL de YouTube**:
input_source = "dialogues"  # @param ["dialogues", "url"]
show_prompt = True  # @param {type:"boolean"}

if input_source == "dialogues":
    fallback_summary = " ".join(translated_texts) if translated_texts else " ".join([r["text"] for r in result])
    selected_url = None
else:
    fallback_summary = None
    selected_url = url

print(f"✨ Generando título viral para TikTok (fuente: {input_source})...\n")

t0 = time.perf_counter()
viral_title = generate_tiktok_title(
    youtube_url=selected_url,
    target_lang="es",
    fallback_text=fallback_summary,
    print_full_prompt=show_prompt
)
t_step4 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 4 completado en {t_step4:.1f}s")
print(f"🔥 Título viral resultante:\n👉 {viral_title}")
print("=" * 60)


In [ ]:
# @title ✂️ 5. Corte de Secuencias de Audio (Vocals EN)
import time
from src.pipeline import split_audio_sequences

# Estructurar segmentos con el texto traducido
combined_segments = []
tr_list = translated_texts if translated_texts else [r["text"] for r in result]
for orig, tr_text in zip(result, tr_list):
    seg = dict(orig)
    seg["text"] = tr_text
    combined_segments.append(seg)

print(f"✂️ Cortando {len(combined_segments)} clips de audio en Seqs_src...\n")
t0 = time.perf_counter()
split_paths = split_audio_sequences(vocals_path, combined_segments)
t_step5 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 5 completado en {t_step5:.1f}s")
print(f"✂️ Archivos generados: {len(split_paths)} en {vocals_path.parent / 'Seqs_src'}")
print("=" * 60)


In [ ]:
# @title 🗣️ 6. Síntesis de Voz TTS y Ajuste de Tempo (Fish Audio + FFmpeg)
import time
from src.audio_utils import clone_voice_segments
from src.tts import model_tts

speaker_audio = video_path.with_name("vocals.wav")
print(f"🗣️ Iniciando síntesis TTS para {len(combined_segments)} segmentos...\n")

t0 = time.perf_counter()
paths_es, times, rate = clone_voice_segments(
    video_path,
    combined_segments,
    "es",
    speaker_audio,
    model_tts,
    slowdown_range=(0.5, 0.8),
    stretch_target_ratio=0.85
)
t_step6 = time.perf_counter() - t0

print("\n" + "=" * 60)
print(f"✅ Paso 6 completado en {t_step6:.1f}s")
print(f"🗣️ Audios generados en Seqs_trg: {len(paths_es)}")
print("=" * 60)


In [ ]:
# @title 🎬 7. Renderizado Final y Benchmark de Parámetros FFmpeg (GPU NVENC vs CPU)
# @markdown Detecta y equipara automáticamente el bitrate del video y audio original (.mp4, .webm, .mkv), compensando códecs de alta eficiencia (VP9/AV1) y preservando el espacio de color.

modo_render = "GPU NVENC (Re-encode)"  # @param ["GPU NVENC (Re-encode)", "Stream Copy (Sin re-encode)", "CPU libx264 (Software)", "Benchmark Comparativo de Presets"]
nvenc_preset = "p4 (Recomendado/Equilibrado)"  # @param ["p1 (Más rápido)", "p2 (Rápido)", "p4 (Recomendado/Equilibrado)", "p6 (Calidad alta)", "p7 (Máxima calidad/Lento)"]
nvenc_multipass = "1 (Quarter-res - Recomendado)"  # @param ["0 (Disabled - Máxima velocidad)", "1 (Quarter-res - Recomendado)", "2 (Full-res - Lento)"]
volumen_musica_fondo = 0.8  # @param {type:"slider", min:0.0, max:1.0, step:0.05} # Multiplicador lineal de amplitud
gpu_device_id = 1  # @param [0, 1] {type:"raw"}
duracion_benchmark_sec = 45  # @param {type:"integer"}

import os, time, subprocess, shutil
from pathlib import Path
import torch, torchaudio
import IPython.display as ipd
from src.audio_utils import merge_cloned_audio
from src.edit_utils import probe_media_profile

# 1. Verificar insumos del pipeline previo
v_path = Path(video_path) if 'video_path' in globals() else None
if not v_path or not v_path.exists():
    raise FileNotFoundError("⚠️ No se encontró video_path. Ejecuta primero las celdas previas de descarga y pipeline.")

job_dir = v_path.parent
no_vocals_file = job_dir / "no_vocals.wav"
seqs_trg_dir = job_dir / "Seqs_trg"

# Detectar perfil multimedia universal (independiente de si es MKV, WEBM o MP4)
prof = probe_media_profile(v_path)
v_target_k = prof["v_bitrate_k"]
a_target_k = prof["a_bitrate_k"]
max_v_k = int(v_target_k * 1.15)
buf_v_k = int(v_target_k * 1.5)
video_dur = float(prof.get("duration", 0.0)) or (v_path.stat().st_size * 8 / (v_target_k * 1000))
video_fps = prof["fps"]

color_args = []
if prof.get("color_range"): color_args += ["-color_range", str(prof["color_range"])]
if prof.get("color_space"): color_args += ["-colorspace", str(prof["color_space"])]
if prof.get("color_primaries"): color_args += ["-color_primaries", str(prof["color_primaries"])]
if prof.get("color_transfer"): color_args += ["-color_trc", str(prof["color_transfer"])]

print("=" * 65)
print(f"🎬 VIDEO BASE: {v_path.name}")
print(f"   • Formato/Códec original: Video: {prof['v_codec'].upper()} | Audio: {prof['a_codec'].upper()}")
print(f"   • Resolución: {prof['width']}x{prof['height']} | FPS: {video_fps}")
print(f"   • Tasa adaptada: Video: {v_target_k} kb/s | Audio: {a_target_k} kb/s")
print("=" * 65)

# 2. Construcción de mezcla de audio en español con torchaudio
audio_mix_out = job_dir / "_dubbed_audio_mix.wav"
def build_dubbed_audio(output_file, max_sec=None):
    sample_rate = rate if 'rate' in globals() and rate else 44100
    
    if 'paths_es' in globals() and 'times' in globals() and paths_es and times:
        vocals_es = merge_cloned_audio(v_path, paths_es, times, sample_rate)
    else:
        vocals_es, sample_rate = torchaudio.load(str(v_path))
        vocals_es[:, :] = 0
    
    if no_vocals_file.exists():
        bg_arr, bg_sr = torchaudio.load(str(no_vocals_file))
        if bg_sr != sample_rate:
            bg_arr = torchaudio.functional.resample(bg_arr, orig_freq=bg_sr, new_freq=sample_rate)
        min_len = min(vocals_es.shape[1], bg_arr.shape[1])
        mixed = vocals_es[:, :min_len] + (bg_arr[:, :min_len] * float(volumen_musica_fondo))
    else:
        mixed = vocals_es
    
    if max_sec:
        max_samples = min(int(max_sec * sample_rate), mixed.shape[1])
        mixed = mixed[:, :max_samples]
    
    max_val = torch.max(torch.abs(mixed))
    if max_val > 1.0:
        mixed = mixed / max_val
    
    torchaudio.save(str(output_file), mixed.cpu(), sample_rate=sample_rate)
    return output_file

# 3. Ejecución según modo seleccionado
preset_clean = nvenc_preset.split(" ")[0].strip()
mp_clean = nvenc_multipass.split(" ")[0].strip()

if modo_render == "Benchmark Comparativo de Presets":
    print(f"🔬 INICIANDO BENCHMARK COMPARATIVO ({duracion_benchmark_sec}s de muestra)...\n")
    bench_audio = job_dir / "_bench_audio.wav"
    build_dubbed_audio(bench_audio, max_sec=duracion_benchmark_sec)

    test_cases = [
        {"name": "NVENC p1 (Max Speed)", "gpu": True, "preset": "p1", "mp": "0"},
        {"name": "NVENC p4 (Equilibrado)", "gpu": True, "preset": "p4", "mp": "1"},
        {"name": "NVENC p7 (Max Quality)", "gpu": True, "preset": "p7", "mp": "2"},
        {"name": "CPU libx264 (Software)", "gpu": False, "preset": "veryfast", "mp": None},
    ]

    results = []
    for tc in test_cases:
        out_test = job_dir / f"_bench_{tc['name'].split()[0].lower()}_{tc['preset']}.mp4"
        cmd = ["ffmpeg", "-y", "-v", "error", "-threads", "0", "-ss", "0", "-t", str(duracion_benchmark_sec), "-i", str(v_path), "-i", str(bench_audio)]
        cmd += ["-map", "0:v:0", "-map", "1:a:0"]
        if tc["gpu"]:
            cmd += ["-gpu", str(gpu_device_id), "-c:v", "h264_nvenc", "-preset", tc["preset"], "-rc", "vbr", "-b:v", f"{v_target_k}k", "-maxrate:v", f"{max_v_k}k", "-bufsize:v", f"{buf_v_k}k"]
            if tc["mp"] != "0":
                cmd += ["-multipass", tc["mp"]]
            cmd += ["-spatial-aq", "1", "-temporal-aq", "1", "-pix_fmt", "yuv420p"] + color_args
        else:
            cmd += ["-c:v", "libx264", "-preset", tc["preset"], "-b:v", f"{v_target_k}k", "-maxrate:v", f"{max_v_k}k", "-bufsize:v", f"{buf_v_k}k", "-pix_fmt", "yuv420p"] + color_args
        cmd += ["-c:a", "aac", "-b:a", f"{a_target_k}k", str(out_test)]

        t_start = time.perf_counter()
        subprocess.run(cmd, check=True)
        t_took = time.perf_counter() - t_start

        size_mb = out_test.stat().st_size / (1024 * 1024) if out_test.exists() else 0.0
        fps_render = (duracion_benchmark_sec * video_fps) / t_took if t_took > 0 else 0
        rel_speed = duracion_benchmark_sec / t_took if t_took > 0 else 0
        bitrate_k = int((size_mb * 8 * 1024) / duracion_benchmark_sec) if duracion_benchmark_sec > 0 else 0

        results.append({
            "nombre": tc["name"],
            "tiempo": t_took,
            "fps": fps_render,
            "speedup": rel_speed,
            "size": size_mb,
            "bitrate": bitrate_k
        })

    print("\n" + "=" * 75)
    print(f"📊 TABLA COMPARATIVA DE RENDIMIENTO (Tasa adaptada: {v_target_k} kbps):")
    print("=" * 75)
    print(f"{'Configuración':<24} | {'Tiempo':<8} | {'FPS':<9} | {'Velocidad':<10} | {'Tamaño':<8} | {'Bitrate':<9}")
    print("-" * 75)
    for r in results:
        print(f"{r['nombre']:<24} | {r['tiempo']:>6.2f}s | {r['fps']:>7.1f} | {r['speedup']:>8.1f}x | {r['size']:>6.1f}MB | {r['bitrate']:>7}k")
    print("=" * 75)
    print("💡 CONCLUSIÓN TÉCNICA:")
    print("   • Todos los presets conservan ahora exactamente el peso del original.")
    print("   • 'p4' es el punto óptimo recomendado para Turing: calidad idéntica a p7 al doble o triple de FPS.")

else:
    # Renderizado completo del video
    out_render = job_dir / f"final_{modo_render.split()[0].lower()}.mp4"
    print(f"🚀 Preparando renderizado completo ({video_dur:.1f}s)...\n")
    build_dubbed_audio(audio_mix_out)

    if modo_render == "Stream Copy (Sin re-encode)":
        cmd = ["ffmpeg", "-y", "-v", "error", "-i", str(v_path), "-i", str(audio_mix_out)]
        cmd += ["-map", "0:v:0", "-c:v", "copy", "-map", "1:a:0", "-c:a", "aac", "-b:a", f"{a_target_k}k", str(out_render)]
    elif modo_render == "GPU NVENC (Re-encode)":
        cmd = ["ffmpeg", "-y", "-v", "error", "-threads", "0", "-i", str(v_path), "-i", str(audio_mix_out)]
        cmd += ["-map", "0:v:0", "-map", "1:a:0", "-gpu", str(gpu_device_id), "-c:v", "h264_nvenc"]
        cmd += ["-preset", preset_clean, "-rc", "vbr", "-b:v", f"{v_target_k}k", "-maxrate:v", f"{max_v_k}k", "-bufsize:v", f"{buf_v_k}k"]
        if mp_clean != "0":
            cmd += ["-multipass", mp_clean]
        cmd += ["-spatial-aq", "1", "-temporal-aq", "1", "-pix_fmt", "yuv420p"] + color_args + ["-c:a", "aac", "-b:a", f"{a_target_k}k", str(out_render)]
    else:  # CPU libx264
        cmd = ["ffmpeg", "-y", "-v", "error", "-threads", "0", "-i", str(v_path), "-i", str(audio_mix_out)]
        cmd += ["-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-preset", "veryfast", "-b:v", f"{v_target_k}k", "-maxrate:v", f"{max_v_k}k", "-bufsize:v", f"{buf_v_k}k", "-pix_fmt", "yuv420p"] + color_args + ["-c:a", "aac", "-b:a", f"{a_target_k}k", str(out_render)]

    t_start = time.perf_counter()
    subprocess.run(cmd, check=True)
    t_elapsed = time.perf_counter() - t_start

    size_mb = out_render.stat().st_size / (1024 * 1024)
    speedup = video_dur / t_elapsed if t_elapsed > 0 else 0

    print("=" * 65)
    print("🏁 EXPORTACIÓN FINALIZADA CON ÉXITO:")
    print(f"• Archivo generado: {out_render.name} ({size_mb:.1f} MB)")
    print(f"• Modo utilizado: {modo_render}")
    print(f"• Tiempo de procesamiento: {t_elapsed:.1f} segundos ({speedup:.1f}x tiempo real)")
    print("=" * 65)

    if out_render.exists():
        ipd.display(ipd.Video(str(out_render), embed=True, width=540))
